# CSV-vs-Parquet

In [3]:
# Making sure to link pyspark to the right Spark folder with findspark
import findspark
import time
import pandas as pd
from pathlib import Path
from functools import wraps
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
findspark.init('/opt/spark')

In [4]:
conf = SparkConf().setAppName("csv-vs-parquet")
sc = SparkContext(conf=conf)

26/05/13 02:03:57 WARN spark.SparkContext: Another SparkContext is being constructed (or threw an exception in its constructor).  This may indicate an error, since only one SparkContext may be running in this JVM (see SPARK-2243). The other SparkContext was created at:
org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
sun.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
sun.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:62)
sun.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.lang.reflect.Constructor.newInstance(Constructor.java:423)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:357)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)
py4j.Gatewa

In [5]:
! hadoop fs -put ../datasets/F1

26/05/13 02:04:34 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
F1_LAP_TIMES_PATH = "hdfs://node-master:9000/user/root/F1/lapTimes.csv"

In [7]:
F1_LAP_TIMES_DEST_PATH = "hdfs://node-master:9000/user/root/F1/lapTimes.parquet"

In [8]:
F1_LAP_TIMES_DEST_PATH_2 = "hdfs://node-master:9000/user/root/F1/lapTimes2.parquet"

In [9]:
! hadoop fs -ls /user/root/F1/

26/05/13 02:04:38 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Found 13 items
-rw-r--r--   2 root supergroup       8667 2026-05-13 02:04 /user/root/F1/circuits.csv
-rw-r--r--   2 root supergroup     224140 2026-05-13 02:04 /user/root/F1/constructorResults.csv
-rw-r--r--   2 root supergroup     267256 2026-05-13 02:04 /user/root/F1/constructorStandings.csv
-rw-r--r--   2 root supergroup      15617 2026-05-13 02:04 /user/root/F1/constructors.csv
-rw-r--r--   2 root supergroup     768136 2026-05-13 02:04 /user/root/F1/driverStandings.csv
-rw-r--r--   2 root supergroup      79533 2026-05-13 02:04 /user/root/F1/drivers.csv
-rw-r--r--   2 root supergroup   12118621 2026-05-13 02:04 /user/root/F1/lapTimes.csv
-rw-r--r--   2 root supergroup     220898 2026-05-13 02:04 /user/root/F1/pitStops.csv
-rw-r--r--   2 root supergroup     315477 2026-05-13 02:04 /user/root/F1/qualifying.csv
-rw-r--r--   2 root supergrou

In [11]:
! hadoop fs -du -h /user/root/F1/lapTimes.csv

26/05/13 02:04:53 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
11.6 M  /user/root/F1/lapTimes.csv


## Pandas

In [12]:
! pip install pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.8/38.8 MB 7.6 MB/s eta 0:00:00a 0:00:01


In [13]:
filepath = Path("../datasets/F1/lapTimes.csv")

In [14]:
dest_folder = Path("../datasets/F1")

In [15]:
filepath.exists()

True

In [16]:
fields = [
    "raceId",
    "driverId",
    "lap",
    "position",
    "time",
    "milliseconds",
]

In [17]:
! pip freeze | grep pandas

pandas==2.0.3


In [18]:
df = pd.read_csv(filepath, dtype={field: str for field in fields})

In [19]:
df.dtypes

raceId          object
driverId        object
lap             object
position        object
time            object
milliseconds    object
dtype: object

In [20]:
df.iloc[0]

raceId               841
driverId              20
lap                    1
position               1
time            1:38.109
milliseconds       98109
Name: 0, dtype: object

In [21]:
df["raceId"] = df["raceId"].str.replace(r'\D+', '', regex=True).astype('int')

In [22]:
df["driverId"] = df["driverId"].str.replace(r'\D+', '', regex=True).astype('int')
df["lap"] = df["lap"].str.replace(r'\D+', '', regex=True).astype('int')
df["position"] = df["position"].str.replace(r'\D+', '', regex=True).astype('int')
df["milliseconds"] = df["milliseconds"].str.replace(r'\D+', '', regex=True).astype('int')

In [23]:
df.dtypes

raceId           int64
driverId         int64
lap              int64
position         int64
time            object
milliseconds     int64
dtype: object

In [24]:
df.to_parquet(dest_folder / "lapTimes.parquet")

## DataFrame

In [25]:
spark = SparkSession(sc)

In [26]:
dflt = spark.read.format("csv").option("header", "true").load(F1_LAP_TIMES_PATH)

In [53]:
type(dflt.head(1)[0]["raceId"])

str

In [56]:
schema = StructType([
    StructField("raceId", IntegerType(), True),
    StructField("driverId", IntegerType(), True),
    StructField("lap", IntegerType(), True),
    StructField("position", IntegerType(), True),
    StructField("time", StringType(), True),
    StructField("milliseconds", IntegerType(), True),
])

In [57]:
dflt2 = spark.read.format("csv").option("header", "true").schema(schema).load(F1_LAP_TIMES_PATH)

In [58]:
dflt2.head(1)

[Row(raceId=841, driverId=20, lap=1, position=1, time='1:38.109', milliseconds=98109)]

In [65]:
dflt2.write.parquet(F1_LAP_TIMES_DEST_PATH)

In [61]:
! hadoop fs -du -h /user/root/f1/lapTimes.parquet

23/08/03 00:38:19 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
0      /user/root/f1/lapTimes.parquet/_SUCCESS
1.8 M  /user/root/f1/lapTimes.parquet/part-00000-6f9a116b-98ea-45a2-a81b-092a5b6cc4e7-c000.snappy.parquet
1.1 M  /user/root/f1/lapTimes.parquet/part-00001-6f9a116b-98ea-45a2-a81b-092a5b6cc4e7-c000.snappy.parquet


In [ ]:
dflt2.coalesce(1).write.parquet(F1_LAP_TIMES_DEST_PATH_2)

In [66]:
! hadoop fs -du -h /user/root/f1/lapTimes2.parquet

23/08/03 00:41:13 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
0      /user/root/f1/lapTimes2.parquet/_SUCCESS
2.5 M  /user/root/f1/lapTimes2.parquet/part-00000-7bdf9fdc-635c-4ee2-943f-f54e15faf910-c000.snappy.parquet


In [11]:
dfr.join(
    dfd, dfr.driverId == dfd.driverId, "inner"
).groupBy(
    dfr.driverId, dfd.driverRef
).agg(
    count(dfr.raceId).alias("races")
).orderBy(
    col("races").desc()
).limit(10).show()

+--------+------------------+-----+
|driverId|         driverRef|races|
+--------+------------------+-----+
|      22|       barrichello|  326|
|      18|            button|  309|
|      30|michael_schumacher|  308|
|       4|            alonso|  293|
|       8|         raikkonen|  273|
|      13|             massa|  271|
|     119|           patrese|  257|
|      15|            trulli|  256|
|      14|         coulthard|  247|
|      21|        fisichella|  231|
+--------+------------------+-----+



In [12]:
dfr.registerTempTable("races")

In [13]:
dfd.registerTempTable("drivers")

In [14]:
spark.sql("""
select r.driverId, d.driverRef, count(0) races
from races r
  inner join drivers d on r.driverId = d.driverId
group by r.driverId, d.driverRef
order by races desc
limit 10
""").show()

+--------+------------------+-----+
|driverId|         driverRef|races|
+--------+------------------+-----+
|      22|       barrichello|  326|
|      18|            button|  309|
|      30|michael_schumacher|  308|
|       4|            alonso|  293|
|       8|         raikkonen|  273|
|      13|             massa|  271|
|     119|           patrese|  257|
|      15|            trulli|  256|
|      14|         coulthard|  247|
|      21|        fisichella|  231|
+--------+------------------+-----+

